In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool()
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for the information"""
    return tavily_client.search(query)

In [9]:
from ipywidgets import FileUpload
from IPython.display import display

uploader= FileUpload(accept=".jpg", multiple=False)
display(uploader)

FileUpload(value=(), accept='.jpg', description='Upload')

In [10]:
print(uploader.value)

({'name': 'images (1).jpg', 'type': 'image/jpeg', 'size': 45276, 'content': <memory at 0x000001B4A0693B80>, 'last_modified': datetime.datetime(2026, 6, 19, 7, 7, 5, 360000, tzinfo=datetime.timezone.utc)},)


In [11]:
import base64
uploaded_file = uploader.value[0]

content_mv = uploaded_file["content"]

img_bytes = bytes(content_mv)

img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [12]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house,

or image of his/her fridge.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [13]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model='gpt-5-nano',
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [15]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "what i can build from these ingredients"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]},
    config
)

print(response['messages'][-1].content)

Nice pantry to work with. Here are tasty, doable ideas using broccoli, bell peppers, mushrooms, onions, tomatoes, and a few common staples (garlic, olive oil, lemon, etc.). I paired each idea with a quick note on how your ingredients map to it and links to the full recipes if you’d like.

1) Sheet-Pan Roasted Vegetables
- What you’ll use: broccoli, mushrooms, bell peppers, onions, tomatoes, garlic, olive oil, salt/pepper.
- Why it’s great: simple one-pan side or base bowl; you can drizzle with lemon at the end for brightness.
- Link: Sheet Pan Roasted Vegetables (For the Love of Cooking)
  https://fortheloveofcooking.net/2021/03/sheet-pan-roasted-vegetables.html

2) Mushroom, Broccoli & Pepper Stir-Fry
- What you’ll use: broccoli, mushrooms, bell peppers, onion, garlic; finish with soy sauce or tamari.
- Why it’s great: quick weeknight dinner; serve with rice or noodles.
- Link: Mushroom Stir Fry with Broccoli & Pepper
  https://www.lastfoodblog.com/mushroom-stir-fry-with-broccoli-pepp